In [1]:
import pandas as pd
from lightfm import LightFM
import rectools
from rectools.models import LightFMWrapperModel
from rectools.dataset import Dataset
from rectools.models.utils import recommend_from_scores
from tqdm.auto import tqdm

from libs.DataLoader import Loader
from libs.utils import *

C:\Users\demid\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightfm\_lightfm_fast.py:9: UserWarning: LightFM was compiled without OpenMP support. Only a single thread will be used.
  warnings.warn(


In [2]:
loader = Loader('ur0.01_ir0.01', 64)
(train_df, val_df), users_df, items_df = Loader.load_data()

train_df = train_df[train_df[TARGET] != 0]
val_df = val_df[val_df[TARGET] == 1]

print(f"Train: {len(train_df)} Val: {len(val_df)}")

Train: 172235 Val: 8179


In [3]:
for df in [train_df, val_df]:
    df["weight"] = 10
    df["datetime"] = 0

embeddings = items_df[EMBEDDING].apply(pd.Series)
embeddings.columns = [f"{EMBEDDING}_{i}" for i in embeddings.columns]
items_df = pd.concat([items_df.drop(columns=[EMBEDDING, "author_id"]), embeddings], axis=1)

In [1]:
dataset = Dataset.construct(
    train_df,
    user_features_df=users_df, cat_user_features=USERS_CAT_FEATURES, make_dense_user_features=True,
    item_features_df=items_df, cat_item_features=ITEMS_CAT_FEATURES, make_dense_item_features=True,
)

model = LightFMWrapperModel(
    LightFM(no_components=10, loss="logistic", random_state=RANDOM_STATE),
    epochs=10,
    verbose=1,
)

NameError: name 'Dataset' is not defined

In [ ]:
model.fit(dataset)

Epoch:   0%|          | 0/10 [00:00<?, ?it/s]

In [8]:
def recommend_users_for_cold_items_manual(
        model: rectools.models.LightFMWrapperModel,
        dataset: Dataset,
        item_ids: list,
        k: int = 10,
        only_warm_users: bool = True
) -> pd.DataFrame:
    assert model.is_fitted

    # Получаем факторы пользователей и товаров
    user_factors = model._get_users_factors(dataset)
    item_factors = model._get_items_factors(dataset)

    # user_factors.embeddings: (n_users, no_components)
    # user_factors.biases: (n_users,)
    # item_factors.embeddings: (n_items, no_components)
    # item_factors.biases: (n_items,)

    user_embeddings = user_factors.embeddings
    user_biases = user_factors.biases
    item_embeddings = item_factors.embeddings
    item_biases = item_factors.biases

    # Определяем каких пользователей рекомендовать
    if only_warm_users:
        user_indices = np.arange(dataset.n_hot_users)  # Только теплые пользователи
    else:
        user_indices = np.arange(len(user_embeddings))  # Все пользователи

    recommendations = []

    for item_id in tqdm(item_ids):
        # Получаем внутренний индекс холодного товара
        try:
            item_idx = dataset.item_id_map.convert_to_internal([item_id])
        except KeyError:
            print(f"Товар {item_id} не найден в dataset, пропускаем")
            continue

        # Получаем embedding и bias для холодного товара
        item_embedding = item_embeddings[item_idx][0]
        item_bias = item_biases[item_idx] if item_biases is not None else 0

        # Вычисляем скоры для всех выбранных пользователей
        # score = user_embedding @ item_embedding + user_bias + item_bias
        user_embs_subset = user_embeddings[user_indices]
        user_biases_subset = user_biases[user_indices] if user_biases is not None else np.zeros(len(user_indices))

        # Матричное умножение + biases
        dot_products = user_embs_subset @ item_embedding
        scores = dot_products + user_biases_subset + item_bias

        # Получаем топ-K пользователей
        top_user_indices, top_scores = recommend_from_scores(
            scores,
            k=k,
            sorted_whitelist=None
        )

        recommendations.append({
            ITEM: item_id,
            USER+"s": dataset.user_id_map.convert_to_external(
                list(map(lambda x: user_indices[x], top_user_indices))
            ),
        })
        # Преобразуем внутренние индексы в исходные ID
    return pd.DataFrame(recommendations)

In [9]:
val_items = list(set(val_df[ITEM]) & set(items_df[ITEM]))
cold_val_items = list(set(val_df[ITEM]) & (set(items_df[ITEM]) - set(train_df[ITEM])))

In [10]:
predict = recommend_users_for_cold_items_manual(
    model,
    dataset,
    val_items,
    k=100,
    only_warm_users=True,
)

  0%|          | 0/2336 [00:00<?, ?it/s]

In [11]:
(NDCG(predict, val_df[val_df[ITEM].isin(val_items)].groupby(ITEM).agg(user_ids=(USER, list))),
 NDCG(predict, val_df[val_df[ITEM].isin(cold_val_items)].groupby(ITEM).agg(user_ids=(USER, list))))

(np.float64(0.00039736445127743994), np.float64(0.0004924264790216768))